# Phase 7 — Validation Model Evaluation

This notebook reloads the untuned CatBoost, LightGBM, and XGBoost artifacts from Phase 6 and evaluates them in detail on the fixed 2,287 validation rows. It does **not** retrain the models, tune hyperparameters, or evaluate the test set.

The goal is to understand not only which model has the best overall score, but also where each model works well or fails.

## 1. Imports and project paths

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)
pd.set_option('display.max_rows', 30)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    MODEL_EVALUATION_REPORT_PATH, MODEL_EVALUATION_SUMMARY_PATH,
    PROCESSED_DATA_PATH, SPLIT_ASSIGNMENT_PATH, TABLES_DIR,
)
from src.evaluate_models import (
    BRAND_METRICS_FILENAME, FUEL_METRICS_FILENAME, LARGEST_ERRORS_FILENAME,
    OVERALL_METRICS_FILENAME, PRICE_BAND_METRICS_FILENAME,
    TRANSMISSION_METRICS_FILENAME, run_model_evaluation,
)
from src.validate_data import file_sha256

print(f'Project root: {PROJECT_ROOT}')
print(f'Cleaned input: {PROCESSED_DATA_PATH}')
print(f'Fixed split: {SPLIT_ASSIGNMENT_PATH}')

## 2. Protect the evaluation design

The hashes identify the exact cleaned dataset and split assignment. Phase 7 will evaluate only rows labelled `validation`. The test target stays unavailable.

In [ ]:
cleaned_hash_before = file_sha256(PROCESSED_DATA_PATH)
split_hash_before = file_sha256(SPLIT_ASSIGNMENT_PATH)
assignment = pd.read_csv(SPLIT_ASSIGNMENT_PATH)

display(assignment['split'].value_counts().reindex(['train', 'validation', 'test']).rename('rows').to_frame())
print(f'Rows available to Phase 7 evaluation: {int(assignment["split"].eq("validation").sum()):,}')
print('Test rows evaluated: 0')
print(f'Cleaned SHA-256: {cleaned_hash_before}')
print(f'Split SHA-256:   {split_hash_before}')

## 3. Reload and evaluate the saved artifacts

This reusable workflow reloads all three models and reproduces their Phase 6 validation predictions. An exact match confirms that the saved artifacts behave like the models that produced the recorded metrics. No model is trained in this phase.

In [ ]:
phase7_summary = run_model_evaluation()
verification = phase7_summary['phase6_artifact_verification']

print(f"Predictions reproduced: {verification['predictions_reproduced']}")
print(f"Maximum prediction difference: ₹{verification['maximum_absolute_prediction_difference_inr']:.8f}")
print(f"Models retrained: {verification['models_retrained']}")
print(f"Test set evaluated: {phase7_summary['test_set']['evaluated']}")

## 4. Overall validation results

All required metrics are calculated on the original rupee scale after the log-price predictions are converted back with `expm1`.

The residual definition is:

`residual = actual price − predicted price`

A positive residual means underprediction; a negative residual means overprediction.

In [ ]:
overall = pd.read_csv(TABLES_DIR / OVERALL_METRICS_FILENAME)
overall_display = overall.copy()
for column in ['mae_inr', 'rmse_inr', 'median_absolute_error_inr', 'mean_residual_inr', 'p90_absolute_error_inr']:
    overall_display[column] = overall_display[column].map(lambda value: f'{"-" if value < 0 else ""}₹{abs(value):,.0f}')
for column in ['within_10_percent_rate', 'within_20_percent_rate']:
    overall_display[column] = overall_display[column].map(lambda value: f'{value:.1%}')
overall_display['r2'] = overall_display['r2'].map(lambda value: f'{value:.4f}')
overall_display['rmsle'] = overall_display['rmsle'].map(lambda value: f'{value:.4f}')
display(overall_display)

catboost = overall.loc[overall['model'].eq('CatBoost')].iloc[0]
print(f"CatBoost MAE: ₹{catboost['mae_inr']:,.0f}")
print(f"CatBoost predictions within ±20%: {catboost['within_20_percent_rate']:.1%}")

## 5. Actual-versus-predicted and residual diagnostics

Points close to the diagonal are more accurate. Logarithmic axes keep inexpensive and luxury listings visible together.

![Actual versus predicted](../reports/figures/16_actual_vs_predicted.png)

The central residual distributions are similar, but the largest expensive-car misses differ substantially.

![Residual distribution](../reports/figures/17_residual_distribution.png)

The widening funnel shows that rupee errors increase with vehicle price.

![Residuals versus predicted price](../reports/figures/18_residuals_vs_predicted.png)

## 6. Price-band stability

A model that performs well overall can still fail in a market segment. CatBoost has the lowest MAE in three of four price bands; XGBoost is narrowly best in the ₹5–10 lakh band.

In [ ]:
bands = pd.read_csv(TABLES_DIR / PRICE_BAND_METRICS_FILENAME)
advanced_bands = bands[bands['model'].ne('Dummy baseline')].copy()
price_band_order = ['Up to ₹5 lakh', '₹5–10 lakh', '₹10–20 lakh', 'Above ₹20 lakh']
advanced_bands['price_band'] = pd.Categorical(advanced_bands['price_band'], categories=price_band_order, ordered=True)
band_winners = advanced_bands.loc[advanced_bands.groupby('price_band', observed=True)['mae_inr'].idxmin(), ['price_band', 'model', 'mae_inr']].sort_values('price_band')
band_winners['mae_inr'] = band_winners['mae_inr'].map(lambda value: f'₹{value:,.0f}')
display(band_winners.reset_index(drop=True))

![Validation MAE by price band](../reports/figures/19_model_mae_by_price_band.png)

## 7. Brand, fuel, and transmission behavior

The brand chart uses a minimum of 30 validation rows so tiny groups are not presented as reliable comparisons. Group MAE also reflects different vehicle and price mixes; it does not mean a brand or transmission *causes* an error.

In [ ]:
brand_metrics = pd.read_csv(TABLES_DIR / BRAND_METRICS_FILENAME)
fuel_metrics = pd.read_csv(TABLES_DIR / FUEL_METRICS_FILENAME)
transmission_metrics = pd.read_csv(TABLES_DIR / TRANSMISSION_METRICS_FILENAME)

major_brands = brand_metrics[brand_metrics['major_brand']].copy()
print(f"Major brands represented: {major_brands['brand'].nunique()}")
print(f"CatBoost major-brand MAE wins: {phase7_summary['major_brand_mae_wins']['CatBoost']}")

fuel_counts = fuel_metrics.groupby('fuel_type')['rows'].first().astype(int).sort_values(ascending=False)
display(fuel_counts.rename('validation_rows').to_frame())
print('Electric and LPG metrics are descriptive only because their validation samples are tiny.')

![Validation MAE by major brand](../reports/figures/20_model_mae_by_major_brand.png)

![Validation MAE by fuel and transmission](../reports/figures/21_model_mae_by_fuel_and_transmission.png)

## 8. Inspect the largest misses

The audit table contains the ten largest underpredictions and overpredictions for every model. These are diagnostic examples, not proof that one listed feature caused the error.

In [ ]:
largest_errors = pd.read_csv(TABLES_DIR / LARGEST_ERRORS_FILENAME)
catboost_errors = largest_errors[largest_errors['evaluated_model'].eq('CatBoost')].copy()
display(catboost_errors[[
    'direction', 'rank', 'brand', 'model', 'vehicle_age', 'km_driven',
    'actual_price', 'predicted_price', 'residual_inr', 'absolute_percentage_error',
]].head(20))

worst_under = catboost_errors.loc[catboost_errors['direction'].eq('Underprediction')].sort_values('rank').iloc[0]
worst_over = catboost_errors.loc[catboost_errors['direction'].eq('Overprediction')].sort_values('rank').iloc[0]
print(f"Largest CatBoost underprediction: {worst_under['brand']} {worst_under['model']} — ₹{worst_under['residual_inr']:,.0f}")
print(f"Largest CatBoost overprediction: {worst_over['brand']} {worst_over['model']} — ₹{abs(worst_over['residual_inr']):,.0f}")

## 9. Phase 7 conclusion and verification

CatBoost should enter Phase 8 as the primary tuning candidate, while LightGBM remains a challenger. This is not final model selection: residual behavior and validation segments support the decision, but the test set remains sealed until the full selection process is complete.

In [ ]:
assert file_sha256(PROCESSED_DATA_PATH) == cleaned_hash_before
assert file_sha256(SPLIT_ASSIGNMENT_PATH) == split_hash_before
assert phase7_summary['phase6_artifact_verification']['predictions_reproduced'] is True
assert phase7_summary['phase6_artifact_verification']['models_retrained'] is False
assert phase7_summary['phase8_recommendation']['primary_tuning_candidate'] == 'CatBoost'
assert phase7_summary['phase8_recommendation']['final_model_selected'] is False
assert phase7_summary['test_set']['evaluated'] is False
assert phase7_summary['test_set']['predictions_generated'] is False
assert phase7_summary['test_set']['metrics_computed'] is False
assert MODEL_EVALUATION_SUMMARY_PATH.exists()
assert MODEL_EVALUATION_REPORT_PATH.exists()

print('Phase 7 verification passed.')
print('Next: Phase 8 will run a controlled, validation-only tuning search.')